# **Problem Statement**

## **Business Context**

Workplace safety in hazardous environments like construction sites and industrial plants is crucial to prevent accidents and injuries. One of the most important safety measures is ensuring workers wear safety helmets, which protect against head injuries from falling objects and machinery. Non-compliance with helmet regulations increases the risk of serious injuries or fatalities, making effective monitoring essential, especially in large-scale operations where manual oversight is prone to errors and inefficiency.

To overcome these challenges, SafeGuard Corp plans to develop an automated image analysis system capable of detecting whether workers are wearing safety helmets. This system will improve safety enforcement, ensuring compliance and reducing the risk of head injuries. By automating helmet monitoring, SafeGuard aims to enhance efficiency, scalability, and accuracy, ultimately fostering a safer work environment while minimizing human error in safety oversight.

## **Objective**

As a data scientist at SafeGuard Corp, you are tasked with developing an image classification model that classifies images into one of two categories:
- **With Helmet:** Workers wearing safety helmets.
- **Without Helmet:** Workers not wearing safety helmets.

## **Data Description**

The dataset consists of **4125 images**, divided into two categories:

- **With Helmet:** 3161 images showing workers wearing helmets.
- **Without Helmet:** 964 images showing workers not wearing helmets.

**Dataset Characteristics:**
- **Variations in Conditions:** Images include diverse environments such as construction sites, factories, and industrial settings, with variations in lighting, angles, and worker postures to simulate real-world conditions.
- **Worker Activities:** Workers are depicted in different actions such as standing, using tools, or moving, ensuring robust model learning for various scenarios.

# **Installing and Importing the Necessary Libraries**

In [ ]:
!pip install tensorflow[and-cuda] scikit-learn==1.6.1 opencv-python==4.12.0.88 seaborn==0.13.2 matplotlib==3.10.0 numpy==2.0.2 pandas==2.2.2 -q

**Note:**

- After running the above cell, kindly restart the notebook kernel (for Jupyter Notebook) or runtime (for Google Colab) and run all cells sequentially from the next cell.

- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import cv2
import math

# TensorFlow / Keras
import keras
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import Adam
from keras.applications.vgg16 import VGG16

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    recall_score, precision_score, f1_score
)

# Display settings
pd.set_option("display.max_columns", None)
warnings = __import__("warnings")
warnings.filterwarnings("ignore")


In [ ]:
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print(tf.__version__)

In [ ]:
# 1. Set Python random seed
random.seed(812)

# 2. Set NumPy random seed
np.random.seed(812)

# 3. Set TensorFlow seed (covers Keras + backend)
tf.keras.utils.set_random_seed(812)

# 4. Enable deterministic GPU ops (if using GPU)
tf.config.experimental.enable_op_determinism()

# **Data Overview**


##Loading the data

In [ ]:
# The project data files should be uploaded to the Colab working directory.
# This helper also allows the notebook to run if it is opened from a local Jupyter directory.

image_candidates = ["/content/images.npy", "images.npy"]
label_candidates = ["/content/labels.csv", "labels.csv"]

image_path = next((p for p in image_candidates if os.path.exists(p)), None)
label_path = next((p for p in label_candidates if os.path.exists(p)), None)

if image_path is None or label_path is None:
    raise FileNotFoundError(
        "Could not find images.npy and/or labels.csv. "
        "Upload both files to the Colab session before running this cell."
    )

images = np.load(image_path)
labels = pd.read_csv(label_path)

print("Images shape :", images.shape)
print("Labels shape :", labels.shape)
print("Image dtype  :", images.dtype)
print("Pixel range  :", images.min(), "to", images.max())
print("\nLabel counts:")
print(labels["label"].value_counts().sort_index())


# **Exploratory Data Analysis**

###Plot random images from each of the classes and print their corresponding labels.

In [ ]:
# Plot multiple random examples from both classes.
label_names = {0: "Without Helmet", 1: "With Helmet"}
y_all = labels["label"].astype(int)

fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for row, class_id in enumerate([0, 1]):
    class_indices = np.where(y_all.to_numpy() == class_id)[0]
    chosen = np.random.choice(class_indices, size=4, replace=False)

    for col, idx in enumerate(chosen):
        axes[row, col].imshow(images[idx])
        axes[row, col].set_title(f"{label_names[class_id]} | Label: {class_id}")
        axes[row, col].axis("off")

plt.suptitle("Random Images from Each Helmet-Compliance Class", fontsize=14)
plt.tight_layout()
plt.show()


## Checking for class imbalance


In [ ]:
class_counts = y_all.value_counts().sort_index()
class_percent = (class_counts / len(y_all) * 100).round(2)

class_summary = pd.DataFrame({
    "Class": [label_names[i] for i in class_counts.index],
    "Count": class_counts.values,
    "Percentage": class_percent.values
}, index=class_counts.index)

display(class_summary)

plt.figure(figsize=(7, 4))
ax = sns.countplot(x=y_all, order=[0, 1])
plt.title("Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks([0, 1], ["Without Helmet (0)", "With Helmet (1)"])

for p in ax.patches:
    ax.annotate(
        f"{int(p.get_height())}",
        (p.get_x() + p.get_width()/2, p.get_height()),
        ha="center", va="bottom"
    )

plt.tight_layout()
plt.show()

imbalance_ratio = class_counts.max() / class_counts.min()
print(f"Majority-to-minority ratio: {imbalance_ratio:.2f}:1")


### **EDA Observations**

- The dataset contains **4,125 RGB images** of size **200 × 200 × 3**, with pixel values in the standard 0–255 range.
- The visual samples show meaningful real-world variation in worker appearance, background, pose, viewing angle, and workplace setting. This makes the task more representative than a highly controlled image dataset.
- The target is **imbalanced**: **3,161 images (76.63%)** belong to the *With Helmet* class, whereas **964 images (23.37%)** belong to the *Without Helmet* class. The majority class is therefore about **3.28 times** the minority class.
- Because of this imbalance, **accuracy alone can be misleading**. In addition to overall accuracy, precision, recall and F1-score, the evaluation will specifically monitor **recall for the Without Helmet class**. Missing a worker who is not wearing a helmet is the more safety-critical error because the system would incorrectly treat a non-compliant situation as compliant.
- The train, validation and test partitions should therefore be **stratified** so that the same class proportions are maintained across all three datasets.


# **Data Preprocessing**

### Splitting the dataset



In [ ]:
# Use a stratified 70% / 15% / 15% split.
# The test set remains untouched until the final model has been selected.

X_train, X_temp, y_train, y_temp = train_test_split(
    images,
    y_all,
    test_size=0.30,
    random_state=812,
    stratify=y_all
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=812,
    stratify=y_temp
)

print("Training set  :", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set      :", X_test.shape, y_test.shape)

split_distribution = pd.DataFrame({
    "Train %": y_train.value_counts(normalize=True).sort_index() * 100,
    "Validation %": y_val.value_counts(normalize=True).sort_index() * 100,
    "Test %": y_test.value_counts(normalize=True).sort_index() * 100
}).round(2)

split_distribution.index = ["Without Helmet (0)", "With Helmet (1)"]
print("\nClass proportions after stratified splitting:")
display(split_distribution)


### Data Normalization

In [ ]:
# Normalize image pixels from [0, 255] to [0, 1].
X_train_normalized = X_train.astype("float32") / 255.0
X_val_normalized = X_val.astype("float32") / 255.0
X_test_normalized = X_test.astype("float32") / 255.0

print("Training normalized range  :", X_train_normalized.min(), "to", X_train_normalized.max())
print("Validation normalized range:", X_val_normalized.min(), "to", X_val_normalized.max())
print("Test normalized range      :", X_test_normalized.min(), "to", X_test_normalized.max())


### **Pre-processing Observations**

- A **stratified 70% / 15% / 15% split** is used so that training, validation and test sets retain approximately the same helmet/no-helmet proportions as the full dataset.
- The test set is kept separate from model development and is only used once, after final model selection.
- Pixel values are normalized from **0–255 to 0–1**, which improves numerical stability during neural-network training.
- No data augmentation is applied to validation or test data. Augmentation is introduced only for Model 4 and only on the training set.


# **Model Building**

## Model Evaluation Criterion

The classes are imbalanced, and the minority class (**Without Helmet = 0**) is the most important class from a workplace-safety perspective. A missed non-compliant worker could result in a safety risk being overlooked.

Accordingly, model performance will be assessed using:
- **Accuracy** for overall correctness,
- **Weighted Precision, Recall and F1-score** to summarize performance while accounting for class frequency,
- **Recall for Without Helmet (class 0)** as a safety-focused metric,
- **Confusion matrices** to inspect the types of classification errors.

The **validation set** will be used to compare and select models. The **test set will only be evaluated after final model selection** to provide an unbiased estimate of generalization.


## Utility Functions

In [ ]:
def model_performance_classification(model, predictors, target):
    """
    Compute overall and class-specific classification metrics.
    Class 0 = Without Helmet; Class 1 = With Helmet.
    """
    prob = model.predict(predictors, verbose=0).reshape(-1)
    pred = (prob > 0.5).astype(int)
    target_array = np.asarray(target).reshape(-1)

    return pd.DataFrame({
        "Accuracy": [accuracy_score(target_array, pred)],
        "Recall (Weighted)": [recall_score(target_array, pred, average="weighted", zero_division=0)],
        "Precision (Weighted)": [precision_score(target_array, pred, average="weighted", zero_division=0)],
        "F1 Score (Weighted)": [f1_score(target_array, pred, average="weighted", zero_division=0)],
        "Recall - Without Helmet": [recall_score(target_array, pred, pos_label=0, zero_division=0)],
        "Recall - With Helmet": [recall_score(target_array, pred, pos_label=1, zero_division=0)]
    })


In [ ]:
def plot_confusion_matrix(model, predictors, target):
    """Plot a clearly labelled confusion matrix for helmet compliance."""
    prob = model.predict(predictors, verbose=0).reshape(-1)
    pred = (prob > 0.5).astype(int)
    target_array = np.asarray(target).reshape(-1)

    cm = confusion_matrix(target_array, pred, labels=[0, 1])

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Without Helmet", "With Helmet"],
        yticklabels=["Without Helmet", "With Helmet"]
    )
    plt.xlabel("Predicted Label")
    plt.ylabel("Actual Label")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.show()


##Model 1: Convolutional Neural Network (CNN) from Scratch

In [ ]:
# Model 1: CNN built from scratch
model_1 = Sequential([
    Conv2D(32, (3, 3), activation="relu", padding="same", input_shape=(200, 200, 3)),
    MaxPooling2D((4, 4), padding="same"),

    Conv2D(64, (3, 3), activation="relu", padding="same"),
    MaxPooling2D((2, 2), padding="same"),

    Conv2D(128, (3, 3), activation="relu", padding="same"),

    Flatten(),
    Dense(4, activation="relu"),
    Dense(1, activation="sigmoid")
])

model_1.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_1.summary()

history_1 = model_1.fit(
    X_train_normalized,
    y_train,
    validation_data=(X_val_normalized, y_val),
    epochs=10,
    batch_size=32,
    shuffle=True,
    verbose=1
)

# Training history
plt.figure(figsize=(7, 4))
plt.plot(history_1.history["accuracy"], label="Train Accuracy")
plt.plot(history_1.history["val_accuracy"], label="Validation Accuracy")
plt.title("Model 1 - CNN from Scratch: Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

# Performance
model_1_train_perf = model_performance_classification(model_1, X_train_normalized, y_train)
model_1_valid_perf = model_performance_classification(model_1, X_val_normalized, y_val)

print("Training performance")
display(model_1_train_perf)
plot_confusion_matrix(model_1, X_train_normalized, y_train)

print("Validation performance")
display(model_1_valid_perf)
plot_confusion_matrix(model_1, X_val_normalized, y_val)


### Visualizing the predictions

In [ ]:
# Display two validation predictions for Model 1.
sample_indices = [12, 33]

for idx in sample_indices:
    plt.figure(figsize=(2.5, 2.5))
    plt.imshow(X_val[idx])
    plt.axis("off")
    plt.show()

    probability = model_1.predict(X_val_normalized[idx:idx+1], verbose=0)[0][0]
    predicted_label = int(probability > 0.5)

    print("Predicted:", f"{label_names[predicted_label]} ({predicted_label})")
    print("Actual   :", f"{label_names[int(y_val.iloc[idx])]} ({int(y_val.iloc[idx])})")
    print(f"P(With Helmet): {probability:.4f}\n")


### **Model 1 Observations**

- This model learns all visual features **from scratch**, so its validation performance provides the baseline for judging whether transfer learning adds value.
- The training and validation curves should be compared for both the final accuracy and the size of the gap between them. A much higher training score than validation score would indicate overfitting.
- Because the dataset is imbalanced, the most important check is not accuracy alone. The validation confusion matrix and **Recall - Without Helmet** reveal whether the CNN is missing non-compliant workers.
- The validation metrics from this model will be retained for the final model-comparison table.


## Model 2: Transfer Learning with VGG-16 (Base)

In [ ]:
# Model 2: VGG-16 convolutional base + sigmoid output layer
vgg_base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(200, 200, 3)
)

# Freeze all pre-trained convolutional layers.
for layer in vgg_base.layers:
    layer.trainable = False

model_2 = Sequential([
    vgg_base,
    Flatten(),
    Dense(1, activation="sigmoid")
])

model_2.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_2.summary()

epochs = 10
batch_size = 32
plain_datagen = ImageDataGenerator()

history_2 = model_2.fit(
    plain_datagen.flow(
        X_train_normalized,
        y_train,
        batch_size=batch_size,
        seed=812,
        shuffle=True
    ),
    epochs=epochs,
    validation_data=(X_val_normalized, y_val),
    verbose=1
)

plt.figure(figsize=(7, 4))
plt.plot(history_2.history["accuracy"], label="Train Accuracy")
plt.plot(history_2.history["val_accuracy"], label="Validation Accuracy")
plt.title("Model 2 - VGG-16 Base: Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

model_2_train_perf = model_performance_classification(model_2, X_train_normalized, y_train)
model_2_valid_perf = model_performance_classification(model_2, X_val_normalized, y_val)

print("Training performance")
display(model_2_train_perf)
plot_confusion_matrix(model_2, X_train_normalized, y_train)

print("Validation performance")
display(model_2_valid_perf)
plot_confusion_matrix(model_2, X_val_normalized, y_val)


### Visualizing the predictions

In [ ]:
# Display two validation predictions for Model 2.
sample_indices = [12, 33]

for idx in sample_indices:
    plt.figure(figsize=(2.5, 2.5))
    plt.imshow(X_val[idx])
    plt.axis("off")
    plt.show()

    probability = model_2.predict(X_val_normalized[idx:idx+1], verbose=0)[0][0]
    predicted_label = int(probability > 0.5)

    print("Predicted:", f"{label_names[predicted_label]} ({predicted_label})")
    print("Actual   :", f"{label_names[int(y_val.iloc[idx])]} ({int(y_val.iloc[idx])})")
    print(f"P(With Helmet): {probability:.4f}\n")


### **Model 2 Observations**

- Model 2 uses the frozen **VGG-16 ImageNet feature extractor**, so only the final classifier is learned from HelmNet data. This substantially reduces the number of trainable parameters compared with training a deep convolutional network from scratch.
- The key comparison with Model 1 is whether the pre-trained visual features improve validation accuracy, weighted F1-score and, most importantly, **recall for workers without helmets**.
- A small train-validation gap would indicate that the frozen VGG-16 representation generalizes well. If both scores are high, it supports transfer learning as an effective approach for this image-classification task.
- The confusion matrix should be inspected carefully for class-0 images predicted as class 1, because these correspond to missed helmet non-compliance.


## Model 3: Transfer Learning with VGG-16 (Base + FFNN)





In [ ]:
# Model 3: VGG-16 base + feed-forward neural network (FFNN)
model_3 = Sequential([
    vgg_base,
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

model_3.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_3.summary()

history_3 = model_3.fit(
    plain_datagen.flow(
        X_train_normalized,
        y_train,
        batch_size=32,
        seed=812,
        shuffle=True
    ),
    epochs=10,
    validation_data=(X_val_normalized, y_val),
    verbose=1
)

plt.figure(figsize=(7, 4))
plt.plot(history_3.history["accuracy"], label="Train Accuracy")
plt.plot(history_3.history["val_accuracy"], label="Validation Accuracy")
plt.title("Model 3 - VGG-16 Base + FFNN: Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

model_3_train_perf = model_performance_classification(model_3, X_train_normalized, y_train)
model_3_valid_perf = model_performance_classification(model_3, X_val_normalized, y_val)

print("Training performance")
display(model_3_train_perf)
plot_confusion_matrix(model_3, X_train_normalized, y_train)

print("Validation performance")
display(model_3_valid_perf)
plot_confusion_matrix(model_3, X_val_normalized, y_val)


#### Visualizing the predictions

In [ ]:
# Display two validation predictions for Model 3.
sample_indices = [12, 33]

for idx in sample_indices:
    plt.figure(figsize=(2.5, 2.5))
    plt.imshow(X_val[idx])
    plt.axis("off")
    plt.show()

    probability = model_3.predict(X_val_normalized[idx:idx+1], verbose=0)[0][0]
    predicted_label = int(probability > 0.5)

    print("Predicted:", f"{label_names[predicted_label]} ({predicted_label})")
    print("Actual   :", f"{label_names[int(y_val.iloc[idx])]} ({int(y_val.iloc[idx])})")
    print(f"P(With Helmet): {probability:.4f}\n")


### **Model 3 Observations**

- Model 3 retains the frozen VGG-16 feature extractor but adds a **128-neuron and 64-neuron FFNN classifier**, together with **50% dropout** to reduce overfitting.
- This model has greater classification capacity than Model 2. The validation results show whether that extra flexibility provides a genuine generalization benefit or simply improves fitting of the training data.
- The train-validation gap should be compared with Model 2. If training performance increases but validation performance does not, the additional dense layers are not adding useful generalization.
- For SafeGuard Corp, the most important comparison remains the model's ability to correctly identify **Without Helmet** cases rather than maximizing overall accuracy alone.


## Model 4: Transfer Learning with VGG-16 (Base + FFNN + Data Augmentation)

- In most of the real-world case studies, it is challenging to acquire a large number of images and then train CNNs.
- To overcome this problem, one approach we might consider is **Data Augmentation**.
- CNNs have the property of **translational invariance**, which means they can recognise an object even if its appearance shifts translationally in some way. - Taking this attribute into account, we can augment the images using the techniques listed below

    -  Horizontal Flip (should be set to True/False)
    -  Vertical Flip (should be set to True/False)
    -  Height Shift (should be between 0 and 1)
    -  Width Shift (should be between 0 and 1)
    -  Rotation (should be between 0 and 180)
    -  Shear (should be between 0 and 1)
    -  Zoom (should be between 0 and 1) etc.

Remember, **data augmentation should not be used in the validation/test data set**.

In [ ]:
# Model 4: VGG-16 base + FFNN + training-only data augmentation
model_4 = Sequential([
    vgg_base,
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

model_4.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_4.summary()

# Use realistic augmentations for worker images.
# Vertical flipping is intentionally avoided because upside-down workers are not a realistic camera condition.
augment_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.10,
    height_shift_range=0.10,
    shear_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True,
    fill_mode="nearest"
)

history_4 = model_4.fit(
    augment_datagen.flow(
        X_train_normalized,
        y_train,
        batch_size=32,
        seed=812,
        shuffle=True
    ),
    epochs=10,
    validation_data=(X_val_normalized, y_val),
    verbose=1
)

plt.figure(figsize=(7, 4))
plt.plot(history_4.history["accuracy"], label="Train Accuracy")
plt.plot(history_4.history["val_accuracy"], label="Validation Accuracy")
plt.title("Model 4 - VGG-16 + FFNN + Data Augmentation: Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

model_4_train_perf = model_performance_classification(model_4, X_train_normalized, y_train)
model_4_valid_perf = model_performance_classification(model_4, X_val_normalized, y_val)

print("Training performance")
display(model_4_train_perf)
plot_confusion_matrix(model_4, X_train_normalized, y_train)

print("Validation performance")
display(model_4_valid_perf)
plot_confusion_matrix(model_4, X_val_normalized, y_val)


#### Visualizing the predictions

In [ ]:
# Display two validation predictions for Model 4.
sample_indices = [12, 33]

for idx in sample_indices:
    plt.figure(figsize=(2.5, 2.5))
    plt.imshow(X_val[idx])
    plt.axis("off")
    plt.show()

    probability = model_4.predict(X_val_normalized[idx:idx+1], verbose=0)[0][0]
    predicted_label = int(probability > 0.5)

    print("Predicted:", f"{label_names[predicted_label]} ({predicted_label})")
    print("Actual   :", f"{label_names[int(y_val.iloc[idx])]} ({int(y_val.iloc[idx])})")
    print(f"P(With Helmet): {probability:.4f}\n")


### **Model 4 Observations**

- Model 4 combines the VGG-16 feature extractor and FFNN classifier with **data augmentation applied only to the training data**. The validation and test images remain unchanged, preventing evaluation leakage.
- Rotation, translation, shear, zoom and horizontal flipping expose the model to plausible variations in camera angle, worker position and scale. This is intended to improve robustness to real workplace imagery.
- Training accuracy may be slightly lower or noisier than Model 3 because every epoch contains modified images. This is not automatically a weakness; the critical question is whether validation performance and **Without Helmet recall** improve or remain more stable.
- If Model 4 achieves validation results comparable to or better than the other models with a small train-validation gap, the augmentation provides a strong practical reason to prefer it for deployment-oriented use.


# **Model Performance Comparison and Final Model Selection**

In [ ]:
# Compare training and validation performance across all four models.
model_names = [
    "CNN from Scratch",
    "VGG-16 Base",
    "VGG-16 Base + FFNN",
    "VGG-16 Base + FFNN + Augmentation"
]

train_perf_list = [
    model_1_train_perf,
    model_2_train_perf,
    model_3_train_perf,
    model_4_train_perf
]

valid_perf_list = [
    model_1_valid_perf,
    model_2_valid_perf,
    model_3_valid_perf,
    model_4_valid_perf
]

models_train_comp_df = pd.concat(train_perf_list, ignore_index=True)
models_train_comp_df.index = model_names

models_valid_comp_df = pd.concat(valid_perf_list, ignore_index=True)
models_valid_comp_df.index = model_names

print("Training Performance Comparison")
display(models_train_comp_df.round(4))

print("\nValidation Performance Comparison")
display(models_valid_comp_df.round(4))

print("\nTrain - Validation Performance Gap")
display((models_train_comp_df - models_valid_comp_df).round(4))

# Safety-first model selection:
# 1) Highest validation recall for Without Helmet
# 2) Highest validation weighted F1
# 3) Highest validation accuracy
ranking = models_valid_comp_df.sort_values(
    by=["Recall - Without Helmet", "F1 Score (Weighted)", "Accuracy"],
    ascending=False
)

top_recall = ranking.iloc[0]["Recall - Without Helmet"]
top_f1 = ranking.iloc[0]["F1 Score (Weighted)"]
top_acc = ranking.iloc[0]["Accuracy"]

# If several models are effectively tied on the three validation criteria,
# prefer the augmented VGG-16 model because it has the strongest robustness-oriented design.
tolerance = 1e-6
tied_models = ranking[
    (np.abs(ranking["Recall - Without Helmet"] - top_recall) <= tolerance) &
    (np.abs(ranking["F1 Score (Weighted)"] - top_f1) <= tolerance) &
    (np.abs(ranking["Accuracy"] - top_acc) <= tolerance)
].index.tolist()

robustness_preference = [
    "VGG-16 Base + FFNN + Augmentation",
    "VGG-16 Base + FFNN",
    "VGG-16 Base",
    "CNN from Scratch"
]

best_model_name = next(
    (name for name in robustness_preference if name in tied_models),
    ranking.index[0]
)

model_lookup = {
    "CNN from Scratch": model_1,
    "VGG-16 Base": model_2,
    "VGG-16 Base + FFNN": model_3,
    "VGG-16 Base + FFNN + Augmentation": model_4
}

best_model = model_lookup[best_model_name]

print("\nSelected Final Model:", best_model_name)
print(
    "Selection basis: validation Recall for Without Helmet first, "
    "then weighted F1 and Accuracy; augmentation/robustness is used only as a tie-breaker."
)


### **Final Model Selection Rationale**

The models are compared using the **validation set only**. This prevents information from the test set influencing model selection.

The primary criterion is **Recall - Without Helmet**, because a false prediction of *With Helmet* for a worker who is actually *Without Helmet* could allow unsafe behavior to go undetected. Weighted F1-score and accuracy are used as secondary criteria so that the selected model still performs well across both classes.

If models are effectively tied on validation performance, preference is given to **VGG-16 + FFNN + Data Augmentation** because the augmentation strategy exposes the model to realistic positional and viewing variations and is therefore the more robustness-oriented architecture for an eventual monitoring system.

The code above prints the model selected from the actual validation results. Only this selected model is evaluated on the test set below.


## Test Performance

In [ ]:
# Final, one-time evaluation on the untouched test set.
model_test_perf = model_performance_classification(
    best_model,
    X_test_normalized,
    y_test
)

print("Final model:", best_model_name)
print("\nTest performance metrics")
display(model_test_perf.round(4))

print("\nTest confusion matrix")
plot_confusion_matrix(best_model, X_test_normalized, y_test)

# Detailed class-wise report for deployment-oriented interpretation.
test_prob = best_model.predict(X_test_normalized, verbose=0).reshape(-1)
test_pred = (test_prob > 0.5).astype(int)

print("\nDetailed test classification report")
print(
    classification_report(
        np.asarray(y_test),
        test_pred,
        labels=[0, 1],
        target_names=["Without Helmet", "With Helmet"],
        digits=4,
        zero_division=0
    )
)


### **Test-Set Interpretation**

The test set is intentionally evaluated **only after the final model has been selected**, so these results provide the cleanest estimate of how the chosen model generalizes to unseen images.

For SafeGuard Corp, the confusion matrix should be read with particular attention to the **actual Without Helmet → predicted With Helmet** cell. These are missed safety violations and represent the most consequential error type.

A deployment candidate should therefore combine:
- strong overall accuracy and weighted F1-score,
- high recall for **Without Helmet**,
- a small number of missed non-compliant workers,
- and stable validation-to-test performance.

Even if the test results are very strong, real-site validation is still required before relying on the system for operational safety decisions.


# **Actionable Insights & Recommendations**

### **Key Takeaways**

- The dataset is meaningfully imbalanced toward workers **with helmets**, so overall accuracy must be interpreted together with class-specific performance.
- Transfer learning with VGG-16 provides a strong way to reuse general visual features rather than learning every feature from scratch.
- The FFNN layers give the classifier additional capacity, while data augmentation is intended to improve robustness to changes in worker position, scale, and camera viewpoint.
- For this safety application, **Recall for Without Helmet** is especially important because a missed non-compliant worker is more consequential than a false alert.
- The final model is selected using validation performance only, and its test performance is then used as the unbiased estimate of generalization.

### **Recommendations for SafeGuard Corp**

1. **Use the selected model as a decision-support component, not the sole safety control.** High model performance can support continuous monitoring, but workplace safety should not depend on an image classifier alone.

2. **Prioritize missed non-compliance in operational monitoring.** Track the rate of actual *Without Helmet* cases predicted as *With Helmet*. This should be a primary production KPI alongside overall accuracy and F1-score.

3. **Introduce confidence-based alerts.** High-confidence no-helmet detections can trigger immediate alerts, while low-confidence or ambiguous cases should be routed for human review.

4. **Validate on real site footage before deployment.** Test the model under low light, glare, shadows, partial occlusion, distant workers, crowded scenes, unusual camera angles, different helmet designs, and indoor/outdoor conditions.

5. **Collect additional minority-class examples.** Because *Without Helmet* is under-represented, future data collection should deliberately include more realistic non-compliance images to improve coverage of the highest-risk class.

6. **Monitor model drift after deployment.** Periodically re-evaluate class-wise precision, recall, F1-score and confusion matrices as sites, cameras, PPE, uniforms and working conditions change.

7. **Consider edge deployment where appropriate.** On-site inference can reduce latency and bandwidth requirements and may improve privacy by avoiding unnecessary transfer of raw video.

8. **Expand gradually to broader PPE monitoring.** After helmet detection is validated in production, the same computer-vision framework could be extended to safety vests, goggles or other required PPE.

### **Conclusion**

The HelmNet analysis demonstrates a structured computer-vision workflow for automated helmet-compliance monitoring, progressing from a CNN built from scratch to transfer-learning models based on VGG-16. The final model should be chosen based on **validation generalization and safety-critical no-helmet recall**, then confirmed on the untouched test set.

This approach aligns the technical evaluation with SafeGuard Corp's real objective: **reducing the chance that unsafe helmet non-compliance goes undetected while enabling scalable, automated safety monitoring.**


<font size=5 color='blue'>Power Ahead!</font>
___